<h2>Walmart_Project</h2>

### Problem Statement
A retail store that has multiple outlets across the country are facing issues in managing the
inventory - to match the demand with respect to supply.
1. You are provided with the weekly sales data for their various outlets. Use statistical
analysis, EDA, outlier analysis, and handle the missing values to come up with various
insights that can give them a clear perspective on the following:

   a. If the weekly sales are affected by the unemployment rate, if yes - which stores
      are suffering the most?

   b. If the weekly sales show a seasonal trend, when and what could be the reason?

   c. Does temperature affect the weekly sales in any manner?

   d. How is the Consumer Price index affecting the weekly sales of various stores?

   e. Top performing stores according to the historical data.

   f. The worst performing store, and how significant is the difference between the
      highest and lowest performing stores.

2. Use predictive modeling techniques to forecast the sales for each store for the next 12
weeks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
walmart = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/Walmart DataSet.csv")
print(walmart)

In [ ]:
# EDA (Exploratory Data Analysis)

In [ ]:
# 1.) Understand the Data

In [ ]:
print(walmart.head())
print(walmart.tail())

In [ ]:
walmart.shape

In [ ]:
walmart.columns

In [ ]:
walmart.describe()

In [ ]:
walmart["Date"] = pd.to_datetime(walmart["Date"], format="%d-%m-%Y")
walmart["Store"] = walmart["Store"].astype("category")

In [ ]:
walmart.info()

In [ ]:
object_cols = walmart.select_dtypes(include=['object'])
for col in object_cols.columns:
    print(f"Value counts for column: {col}")
    print(object_cols[col].value_counts())
    print("\n" + "-"*40 + "\n")

In [ ]:
walmart.dtypes

In [ ]:
# 2.) Handle Missing Values

In [ ]:
walmart.isnull().sum()

In [ ]:
# 2.a.) Handling missing values for Columns

In [ ]:
thresholds_column = {
    "numeric": 50,     # Drop numeric columns with >50% missing
    "datetime": 30,    # Drop datetime columns with >30% missing
    "categorical": 60  # Drop categorical columns with >60% missing
}

In [ ]:
numeric_column = walmart.select_dtypes(include=["number"])
datetime_column = walmart.select_dtypes(include=["datetime"])
categorical_column = walmart.select_dtypes(include=["category"])

In [ ]:
print("Numeric columns:\n", numeric_column.columns.tolist())
print("Datetime columns:\n", datetime_column.columns.tolist())
print("Categorical columns:\n", categorical_column.columns.tolist())

In [ ]:
missing_numeric = numeric_column.isnull().mean() * 100
missing_datetime = datetime_column.isnull().mean() * 100
missing_categorical = categorical_column.isnull().mean() * 100

In [ ]:
numeric_column = numeric_column.loc[:, missing_numeric < thresholds_column["numeric"]]
datetime_column = datetime_column.loc[:, missing_datetime < thresholds_column["datetime"]]
categorical_column = categorical_column.loc[:, missing_categorical < thresholds_column["categorical"]]

In [ ]:
walmart = pd.concat([numeric_column, datetime_column, categorical_column], axis=1)
print(walmart)

In [ ]:
walmart.isnull().sum()

In [ ]:
# 2.b.) Handling missing values for Rows

In [ ]:
row_threshold = 0.3
max_missing_allowed = int(row_threshold * walmart.shape[1])

walmart = walmart[walmart.isna().sum(axis=1) <= max_missing_allowed]

if walmart.isna().values.any():
    print("Some rows still contain missing values.")
else:
    print("No missing values are present in the rows.")

In [ ]:
# Handling remaining missing values

# Fill missing numerical values with median
numerical_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
for col in numerical_cols:
    walmart[col].fillna(walmart[col].median(), inplace=True)

# Fill missing categorical values with mode
categorical_cols = ['Store', 'Holiday_Flag']
for col in categorical_cols:
    walmart[col].fillna(walmart[col].mode()[0], inplace=True)

In [ ]:
# Outlier Analysis
for col in numerical_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(y=walmart[col])
    plt.title(f'Boxplot of {col}')
    plt.show()
for col in numerical_cols:
    Q1 = walmart[col].quantile(0.25)
    Q3 = walmart[col].quantile(0.75)
    IQR = Q3 - Q1
    df = walmart[(walmart[col] >= Q1 - 1.5*IQR) & (walmart[col] <= Q3 + 1.5*IQR)]

In [ ]:
# 1.a.) Weekly Sales vs Unemployment Rate
print("Correlation with Unemployment:", walmart['Weekly_Sales'].corr(walmart['Unemployment']))
store_unemployment = walmart.groupby('Store')[['Weekly_Sales', 'Unemployment']].corr().iloc[0::2,-1]
print(store_unemployment)
for store in walmart['Store'].unique():
    store_data = walmart[walmart['Store'] == store]
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=store_data, x='Unemployment', y='Weekly_Sales')
    plt.title(f'Weekly Sales vs Unemployment - Store {store}')
    plt.show()

In [ ]:
# 1.b.) Seasonal Trend Analysis
walmart.set_index('Date', inplace=True)
plt.figure(figsize=(12,6))
df.groupby('Date')['Weekly_Sales'].sum().plot()
plt.title('Total Weekly Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.show()
monthly_sales = walmart['Weekly_Sales'].resample('M').sum()
monthly_sales.plot(figsize=(12,6), title='Monthly Sales Trend')
plt.show()

In [ ]:
# 1.c.)Temperature vs Weekly Sales
print("Correlation with Temperature:", walmart['Weekly_Sales'].corr(df['Temperature']))
sns.scatterplot(data=walmart, x='Temperature', y='Weekly_Sales')
plt.title('Weekly Sales vs Temperature')
plt.show()

In [ ]:
# 1.d.)CPI vs Weekly Sales
print("Correlation with CPI:", walmart['Weekly_Sales'].corr(walmart['CPI']))
sns.scatterplot(data=walmart, x='CPI', y='Weekly_Sales')
plt.title('Weekly Sales vs CPI')
plt.show()

In [ ]:
# 1.e.&f.) Top and worst performing stores
store_sales = walmart.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)
print("Top Performing Stores:\n", store_sales.head())
print("Worst Performing Stores:\n", store_sales.tail())
diff = store_sales.max() - store_sales.min()
print("Difference between highest and lowest performing stores:", diff)

In [ ]:
# 2.) Predictive modeling
from prophet import Prophet

forecast_horizon = 12  # weeks
for store in walmart['Store'].unique():
    store_df = walmart[walmart['Store'] == store].reset_index()[['Date', 'Weekly_Sales']].rename(columns={'Date':'ds','Weekly_Sales':'y'})
    model = Prophet(weekly_seasonality=True, yearly_seasonality=True)
    model.fit(store_df)
    future = model.make_future_dataframe(periods=forecast_horizon, freq='W')
    forecast = model.predict(future)
    model.plot(forecast)
    plt.title(f'Weekly Sales Forecast - Store {store}')
    plt.show()
    print(forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(forecast_horizon))